In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df=pd.read_csv('/content/drive/MyDrive/brain_train.csv')
df1=pd.read_csv('/content/drive/MyDrive/brain_test.csv')

In [3]:
df.isnull().sum()

,0
Unnamed: 0.1,0
Unnamed: 0,0
0,0
1,0
2,0
...,...
29996,0
29997,0
29998,0
29999,0


In [4]:
df=df.dropna()

In [5]:
df=df.drop(columns=['Unnamed: 0.1','Unnamed: 0'])

In [6]:
df1=df1.drop(columns=['Unnamed: 0.1','Unnamed: 0'])

In [7]:
y_train=df['target']
X_train=df.drop(columns='target')

In [8]:
y_test=df1['target']
X_test=df1.drop(columns='target')

In [9]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [10]:
import numpy as np

X_train = np.asarray(X_train, dtype=np.float32)
X_test = np.asarray(X_test, dtype=np.float32)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Minimum pixel value:", X_train.min())
print("Maximum pixel value:", X_train.max())

# Normalize pixel values only if they are in the 0–255 range
if X_train.max() > 1:
    X_train = X_train / 255.0
    X_test = X_test / 255.0

Training shape: (5602, 30000)
Testing shape: (1600, 30000)
Minimum pixel value: 0.0
Maximum pixel value: 255.0


In [11]:
print(le.classes_)

['glioma' 'meningioma' 'notumor' 'pituitary']


In [12]:
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Input,Dense,Dropout


In [20]:

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Dense, Dropout, BatchNormalization
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau
)

model = Sequential([
    Input(shape=(X_train.shape[1],)),

    Dense(256, activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.4),

    Dense(128, activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.2),

    Dense(4, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7
)

history = model.fit(
    X_train,
    y_train,
    epochs=60,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)


Epoch 1/60
141/141 ━━━━━━━━━━━━━━━━━━━━ 27s 161ms/step - accuracy: 0.4283 - loss: 2.1044 - val_accuracy: 0.4576 - val_loss: 1.8953 - learning_rate: 1.0000e-04
Epoch 2/60
141/141 ━━━━━━━━━━━━━━━━━━━━ 21s 145ms/step - accuracy: 0.5302 - loss: 1.8225 - val_accuracy: 0.4987 - val_loss: 1.6799 - learning_rate: 1.0000e-04
Epoch 3/60
141/141 ━━━━━━━━━━━━━━━━━━━━ 41s 144ms/step - accuracy: 0.5465 - loss: 1.7134 - val_accuracy: 0.5450 - val_loss: 1.5502 - learning_rate: 1.0000e-04
Epoch 4/60
141/141 ━━━━━━━━━━━━━━━━━━━━ 23s 161ms/step - accuracy: 0.5595 - loss: 1.6480 - val_accuracy: 0.5236 - val_loss: 1.6597 - learning_rate: 1.0000e-04
Epoch 5/60
141/141 ━━━━━━━━━━━━━━━━━━━━ 20s 141ms/step - accuracy: 0.5691 - loss: 1.5639 - val_accuracy: 0.4996 - val_loss: 1.6267 - learning_rate: 1.0000e-04
Epoch 6/60
141/141 ━━━━━━━━━━━━━━━━━━━━ 24s 163ms/step - accuracy: 0.5916 - loss: 1.5027 - val_accuracy: 0.4746 - val_loss: 1.6629 - learning_rate: 1.0000e-04
Epoch 7/60
141/141 ━━━━━━━━━━━━━━━━━━━━ 35s 12

In [21]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.5063 - loss: 2.8014
Test Loss: 2.8014206886291504
Test Accuracy: 0.5062500238418579


In [22]:
y_pred = model.predict(X_test)
y_pred = y_pred.argmax(axis=1)

50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step


In [23]:
from sklearn.metrics import classification_report, confusion_matrix

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("Classification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=le.classes_
))

Confusion Matrix:
[[281  71  21  27]
 [ 59 299  19  23]
 [  3   8 205 184]
 [ 90 278   7  25]]
Classification Report:
              precision    recall  f1-score   support

      glioma       0.65      0.70      0.67       400
  meningioma       0.46      0.75      0.57       400
     notumor       0.81      0.51      0.63       400
   pituitary       0.10      0.06      0.08       400

    accuracy                           0.51      1600
   macro avg       0.50      0.51      0.49      1600
weighted avg       0.50      0.51      0.49      1600



In [24]:
model.save("brain_tumor_ann.keras")

In [25]:
from google.colab import files

files.download("brain_tumor_ann.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
import json

classes = le.classes_.tolist()

with open("classes.json", "w") as file:
    json.dump(classes, file)

print(classes)


['glioma', 'meningioma', 'notumor', 'pituitary']


In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.4/270.4 kB 30.3 MB/s eta 0:00:00
